####  Bronze vs Silver Testing – stores Table

This document describes the data quality, integrity, and reconciliation tests performed for the `stores` table during ingestion from Bronze to Silver.





####  Tables Under Test

- Bronze: `coffee.bronze.stores`
- Silver: `coffee.silver.stores`
- Quarantine: `coffee.silver.stores_quarantine`

In [0]:
-- Test 1: Compare row counts between Bronze and Silver
-- Silver count should be <= Bronze count due to filtering and deduplication

SELECT 'bronze' AS layer, COUNT(*) AS record_count
FROM coffee.bronze.stores

UNION ALL

SELECT 'silver' AS layer, COUNT(*) AS record_count
FROM coffee.silver.stores;

In [0]:
-- Test 2: Ensure mandatory columns are NOT NULL in Silver
-- store_id, store_name, city, and state must always be present

SELECT COUNT(*) AS invalid_silver_records
FROM coffee.silver.stores
WHERE
  store_id IS NULL
  OR store_name IS NULL
  OR city IS NULL
  OR state IS NULL;


In [0]:
-- Test 3: All valid Bronze store records should be present in Silver
-- Identifies valid records accidentally dropped

SELECT store_id
FROM coffee.bronze.stores
WHERE
  store_id IS NOT NULL
  AND store_name IS NOT NULL
  AND city IS NOT NULL
  AND state IS NOT NULL

EXCEPT

SELECT store_id
FROM coffee.silver.stores;


In [0]:
-- Test 4: Ensure each store_id appears only once in Silver
-- Confirms deduplication logic is working correctly

SELECT store_id, COUNT(*) AS cnt
FROM coffee.silver.stores
GROUP BY store_id
HAVING COUNT(*) > 1;
